# Tarea A --- Preprocesamiento Tabular

Convierte las variables estructuradas de `data/diagnosticos_itaca_clean.csv`
en arreglos numericos para la rama tabular de la DNN, genera los splits
train/val/test compartidos por todo el equipo y serializa los artefactos
de preprocesamiento (scaler, encoder, mapeo de clases) que usara el backend.

## 0. Configuracion y constantes

In [ ]:
# Librerias
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Rutas como constantes al inicio del script.
# El notebook se ejecuta con working directory = preprocessing/.
CLEAN_DATA_PATH = "../data/diagnosticos_itaca_clean.csv"

SPLITS_DIR = "splits"
ARTIFACTS_DIR = "artifacts"
OUTPUT_DIR = "output"

TRAIN_PATH = f"{SPLITS_DIR}/train.csv"
VAL_PATH = f"{SPLITS_DIR}/val.csv"
TEST_PATH = f"{SPLITS_DIR}/test.csv"

# Semilla fija exigida por el spec en TODA operacion aleatoria.
RANDOM_STATE = 42

# Proporciones de particion definidas en la seccion 4.1 del spec:
# train 75%, val 15%, test 10%. temp = 25% del total; dentro de temp,
# val es 15/25 = 60% de temp y test es 10/25 = 40% de temp.
TRAIN_SIZE = 0.75
VAL_SIZE_OF_TEMP = 0.60

TARGET_COL = "nivel_madurez"

## 4.1 Particiones estratificadas (train / val / test)

Se usa `train_test_split` en dos
pasos, siempre estratificando por `nivel_madurez` para preservar la
proporcion de clases (la clase `Optimizado` es minoritaria, ~13%).

In [ ]:
# Carga del dataset limpio
df = pd.read_csv(CLEAN_DATA_PATH, encoding="utf-8")

print(f"Filas cargadas: {len(df)}")
print(f"Columnas ({len(df.columns)}): {list(df.columns)}")

df.head()

Filas cargadas: 3000
Columnas (8): ['id_diagnostico', 'sector', 'tamano_empresa', 'porcentaje_procesos_documentados', 'presupuesto_anual_tecnologia', 'respuesta_texto', 'nivel_madurez', 'recomendacion_principal']


,id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnologia,respuesta_texto,nivel_madurez,recomendacion_principal
0,DIAG_0000,Tecnologia,Micro,0.05,3000000,"El trabajo es muy empírico, no hay documentaci...",Inicial,Mapear flujos de trabajo de desarrollo y usar ...
1,DIAG_0001,Tecnologia,Grande,0.81,261000000,"Todo está automatizado, usamos datos para mejo...",Optimizado,Aplicar MLOps y analítica predictiva para opti...
2,DIAG_0002,Tecnologia,Pequena,0.27,47000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
3,DIAG_0003,Tecnologia,Pequena,0.30,19000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
4,DIAG_0004,Manufactura,Micro,0.08,6000000,"Todo es manual, dependemos de una sola persona...",Inicial,Estandarizar el registro diario de mermas e in...


In [3]:
# Paso 1: separar train (75%) de un conjunto temporal (25%) que luego se
# divide en val y test. Estratificado por el target para preservar la
# proporcion de clases en cada conjunto.
df_train, df_temp = train_test_split(
    df,
    train_size=TRAIN_SIZE,
    stratify=df[TARGET_COL],
    random_state=RANDOM_STATE,
)

print(f"train: {len(df_train)} filas ({len(df_train) / len(df):.1%})")
print(f"temp:  {len(df_temp)} filas ({len(df_temp) / len(df):.1%})")

train: 2250 filas (75.0%)
temp:  750 filas (25.0%)


In [4]:
# Paso 2: dividir temp (25% del total) en val (60% de temp -> 15% del
# total) y test (40% de temp -> 10% del total). Se estratifica sobre el
# nivel de madurez DENTRO de temp, no sobre el dataset completo.
df_val, df_test = train_test_split(
    df_temp,
    train_size=VAL_SIZE_OF_TEMP,
    stratify=df_temp[TARGET_COL],
    random_state=RANDOM_STATE,
)

print(f"val:  {len(df_val)} filas ({len(df_val) / len(df):.1%})")
print(f"test: {len(df_test)} filas ({len(df_test) / len(df):.1%})")

val:  450 filas (15.0%)
test: 300 filas (10.0%)


### Validaciones

In [5]:
# Criterio de aceptacion 1: los tres splits suman 3000 filas y no
# comparten ningun id_diagnostico entre si.
total_rows = len(df_train) + len(df_val) + len(df_test)
assert total_rows == len(df), f"Los splits no suman el total: {total_rows} != {len(df)}"

ids_train = set(df_train["id_diagnostico"])
ids_val = set(df_val["id_diagnostico"])
ids_test = set(df_test["id_diagnostico"])

assert ids_train.isdisjoint(ids_val), "train y val comparten id_diagnostico"
assert ids_train.isdisjoint(ids_test), "train y test comparten id_diagnostico"
assert ids_val.isdisjoint(ids_test), "val y test comparten id_diagnostico"

print(f"Total filas en los tres splits: {total_rows} (esperado: {len(df)})")
print("Ids sin solapamiento entre splits: OK")

Total filas en los tres splits: 3000 (esperado: 3000)
Ids sin solapamiento entre splits: OK


In [6]:
# Criterio de aceptacion 2: la distribucion porcentual de nivel_madurez
# debe diferir en menos de 1 punto porcentual entre train, val y test
# (evidencia de que la estratificacion funciono).
def class_pct(frame: pd.DataFrame) -> pd.Series:
    return (100 * frame[TARGET_COL].value_counts(normalize=True)).round(2)

pct_table = pd.DataFrame({
    "train": class_pct(df_train),
    "val": class_pct(df_val),
    "test": class_pct(df_test),
}).sort_index()

print("Distribucion porcentual de nivel_madurez por split:")
print(pct_table)

max_diff = (pct_table.max(axis=1) - pct_table.min(axis=1)).max()
print(f"\nMaxima diferencia entre splits: {max_diff:.2f} puntos porcentuales")

assert max_diff < 1.0, "La estratificacion no cumple el criterio de <1pp de diferencia"
print("Estratificacion validada: diferencia < 1 punto porcentual entre splits")

Distribucion porcentual de nivel_madurez por split:
               train    val   test
nivel_madurez                     
Definido       30.36  30.44  30.33
En Desarrollo  30.80  30.67  30.67
Inicial        26.09  26.00  26.33
Optimizado     12.76  12.89  12.67

Maxima diferencia entre splits: 0.33 puntos porcentuales
Estratificacion validada: diferencia < 1 punto porcentual entre splits


### Guardado de los splits

In [7]:
# Guardar los tres splits COMPLETOS (las 8 columnas originales, incluyendo
# id_diagnostico y respuesta_texto que la Tarea B necesita) en UTF-8.
df_train.to_csv(TRAIN_PATH, index=False, encoding="utf-8")
df_val.to_csv(VAL_PATH, index=False, encoding="utf-8")
df_test.to_csv(TEST_PATH, index=False, encoding="utf-8")

print(f"Guardado: {TRAIN_PATH} ({len(df_train)} filas)")
print(f"Guardado: {VAL_PATH} ({len(df_val)} filas)")
print(f"Guardado: {TEST_PATH} ({len(df_test)} filas)")

Guardado: splits/train.csv (2250 filas)
Guardado: splits/val.csv (450 filas)
Guardado: splits/test.csv (300 filas)
